# Libraries

In [ ]:
from ase.visualize import view
from ase.io import read
from ase.data import atomic_numbers, vdw_radii
from ase.build import surface
from ase.build import add_adsorbate
from ase import Atoms
import numpy as np
from typing import List, Tuple, Optional
from scipy.spatial import KDTree
from itertools import combinations
import nglview
from tqdm import tqdm
from scipy.spatial import cKDTree
from scipy.spatial import Delaunay

# Support Functions

In [ ]:
# Supporting Functions (unchanged)
def atomic_radii(structure_path):
    try:
        atoms = read(structure_path)
    except Exception as e:
        raise ValueError(f"Không thể đọc file : {e}")
    
    elements = set(atoms.get_chemical_symbols())
    radii_dict = {}
    for element in elements:
        atomic_number = atomic_numbers[element]
        radius = vdw_radii[atomic_number] if vdw_radii[atomic_number] != 0 else 1.0 
        radii_dict[element] = radius # in angstrom
        print(f"{element} radius: {radius}")
    
    return radii_dict

def molecule_surface_area(atoms, atom_radii, num_points=100000):
    coords_xy = np.array([atom.position[:2] for atom in atoms])
    radii = np.array([atom_radii[atom.symbol] for atom in atoms])
    min_x = np.min(coords_xy[:, 0] - radii)
    max_x = np.max(coords_xy[:, 0] + radii)
    min_y = np.min(coords_xy[:, 1] - radii)
    max_y = np.max(coords_xy[:, 1] + radii)
    box_area = (max_x - min_x) * (max_y - min_y)
    
    points_x = np.random.uniform(min_x, max_x, num_points)
    points_y = np.random.uniform(min_y, max_y, num_points)
    points = np.vstack((points_x, points_y)).T
    
    inside_count = 0
    for i in range(num_points):
        for j, center in enumerate(coords_xy):
            distance = np.sqrt(np.sum((points[i] - center) ** 2))
            if distance <= radii[j]:
                inside_count += 1
                break
    
    area_angstrom2 = (inside_count / num_points) * box_area
    area_nm2 = area_angstrom2 / 100
    print(f"Diện tích bề mặt phân tử (Top view): {area_angstrom2:.2f} Å², {area_nm2:.4f} nm²")
    return area_angstrom2, area_nm2

def compute_distances_vectorized(coords1, coords2):
    diff = coords1[:, np.newaxis] - coords2
    distances = np.sqrt(np.sum(diff**2, axis=2))
    return distances

def rotate_molecule_z_axis(adsorbate):
    rotated_adsorbate = adsorbate.copy()
    pivot = rotated_adsorbate[0].position.copy()
    theta = np.random.uniform(0, 360)
    theta_rad = np.radians(theta)
    
    rotation_matrix = np.array([
        [np.cos(theta_rad), -np.sin(theta_rad), 0],
        [np.sin(theta_rad), np.cos(theta_rad), 0],
        [0, 0, 1]
    ])
    
    rotated_adsorbate.positions -= pivot
    rotated_adsorbate.positions = np.dot(rotated_adsorbate.positions, rotation_matrix.T)
    rotated_adsorbate.positions += pivot
    
    return rotated_adsorbate

# Molecule Functions

In [ ]:
# Precursor/Inhibitor Configuration Functions (unchanged)
def atomic_positions(structure_path):
    atoms = read(structure_path)
    radius = atomic_radii(structure_path)
    print("Atomic symbols:", atoms.get_chemical_symbols())
    print("Atomic positions (Å):")
    print(atoms.positions)
    for symbol, pos in zip(atoms.get_chemical_symbols(), atoms.positions):
        print(f"{symbol}: x={pos[0]:.4f}, y={pos[1]:.4f}, z={pos[2]:.4f}")
    return atoms, radius

def atomic_remove_indices(atoms, remove_indices=None):
    if remove_indices:
        keep_mask = [i not in remove_indices for i in range(len(atoms))]
        atoms = atoms[keep_mask]

    print(f"Sau khi loại bỏ nguyên tử thứ {remove_indices}, cấu trúc phân tử mới là:")
    for i, (s, p) in enumerate(zip(atoms.get_chemical_symbols(), atoms.positions)):
        print(f"{i}: {s} - x={p[0]:.8f}, y={p[1]:.8f}, z={p[2]:.8f}")

    return atoms

def rotate_adsorbate_flexible(adsorbate, ads_site, v1_type='two_atoms', indices=None):
    """Version với hướng mặc định 90 độ"""
    from scipy.spatial.transform import Rotation as R
    import numpy as np
    
    rotated_adsorbate = adsorbate.copy()
    pivot = rotated_adsorbate[0].position.copy()
    
    # Tính v1 như cũ
    if indices is None:
        indices = [1] if v1_type == 'two_atoms' else [1, 2]
    
    if v1_type == 'two_atoms':
        v1_end = rotated_adsorbate.positions[indices[0]]
        v1 = v1_end - pivot
    elif v1_type == 'centroid':
        other_atoms_positions = [rotated_adsorbate.positions[i] for i in indices]
        centroid = np.mean(other_atoms_positions, axis=0)
        v1 = centroid - pivot
    
    # Mặc định v2 là hướng 90 độ (thẳng đứng)
    v2 = np.array([0, 0, 1])  # Vector đơn vị hướng lên
    
    v1 /= np.linalg.norm(v1)
    # v2 đã là unit vector
    
    # Tính rotation
    cross = np.cross(v1, v2)
    dot = np.dot(v1, v2)
    
    if np.linalg.norm(cross) < 1e-8:
        if dot < 0:
            axis = np.array([1, 0, 0])
            angle = np.pi
        else:
            angle = 0.0
            axis = np.array([0, 0, 1])
    else:
        axis = cross / np.linalg.norm(cross)
        angle = np.arccos(np.clip(dot, -1.0, 1.0))
    
    rotation = R.from_rotvec(angle * axis)
    
    # Apply rotation
    rotated_adsorbate.translate(-pivot)
    rotated_adsorbate.positions = rotation.apply(rotated_adsorbate.positions)
    rotated_adsorbate.translate(ads_site)  # Move to ads_site position
    view(rotated_adsorbate)
    print(f"Applied vertical rotation - Angle: {np.degrees(angle):.1f}°")
    print(f"Adsorbate oriented vertically (90°) at position: {ads_site}")
    
    return rotated_adsorbate, angle, axis
'''
def rotate_adsorbate_flexible(adsorbate, ads_site, v1_type='two_atoms', indices=None):
    from scipy.spatial.transform import Rotation as R
    import numpy as np
    
    rotated_adsorbate = adsorbate.copy()
    pivot = rotated_adsorbate[0].position.copy()
    
    if indices is None:
        indices = [1] if v1_type == 'two_atoms' else [1, 2]
    if v1_type == 'two_atoms' and len(indices) != 2:
        indices = [1]
    
    if v1_type == 'two_atoms':
        if len(indices) < 2:
            raise ValueError("Cần ít nhất 2 index cho v1_type 'two_atoms'.")
        v1_start = pivot
        v1_end = rotated_adsorbate.positions[indices[1]]
        v1 = v1_end - v1_start
    elif v1_type == 'centroid':
        if len(indices) < 2:
            raise ValueError("Cần ít nhất 2 index cho v1_type 'centroid'.")
        other_atoms_positions = [rotated_adsorbate.positions[i] for i in indices]
        if len(indices) == 2:
            midpoint = np.mean(other_atoms_positions, axis=0)
            v1 = midpoint - pivot
        else:
            centroid = np.mean(other_atoms_positions, axis=0)
            v1 = centroid - pivot
    else:
        raise ValueError("v1_type phải là 'two_atoms' hoặc 'centroid'.")
    
    v2 = ads_site - pivot
    
    v1 /= np.linalg.norm(v1)
    v2 /= np.linalg.norm(v2)
    
    cross = np.cross(v1, v2)
    dot = np.dot(v1, v2)
    if np.linalg.norm(cross) < 1e-8:
        if dot < 0:
            axis = np.array([1, 0, 0]) if not np.allclose(v1, [1, 0, 0]) else np.array([0, 1, 0])
            angle = np.pi
        else:
            axis = np.array([0, 0, 1])
            angle = 0.0
    else:
        axis = cross / np.linalg.norm(cross)
        angle = np.arccos(np.clip(dot, -1.0, 1.0))
    
    rotation = R.from_rotvec(angle * axis)
    
    rotated_adsorbate.translate(-pivot)
    rotated_adsorbate.positions = rotation.apply(rotated_adsorbate.positions)
    rotated_adsorbate.translate(pivot)
    
    view(rotated_adsorbate)
    print(f"\nApplied rotation - Angle: {np.degrees(angle):.1f}°, Axis: [{axis[0]:.2f}, {axis[1]:.2f}, {axis[2]:.2f}]")
    print(f"v1_type: {v1_type}, Indices used: {indices}")
    
    return rotated_adsorbate, angle, axis
'''

# Slab Functions

In [ ]:
# Substrate Surface Configuration Functions (unchanged)
def generate_slab_from_cif(cif_path, miller_indices=(1, 0, 0), layers=4, vacuum=10.0, size=(1, 1, 1), repeat=(4, 4, 1)):
    structure = read(cif_path)
    slab = surface(structure, miller_indices, layers=layers, vacuum=vacuum)
    slab = slab.repeat(repeat)
    slab.pbc = [True, True, True]
    
    a1 = slab.cell[0]
    a2 = slab.cell[1]
    surface_area_angstrom2 = np.linalg.norm(np.cross(a1, a2))
    surface_area_nm2 = surface_area_angstrom2 / 100
    
    print(f"Slab created with Miller indices {miller_indices} and {len(slab)} atoms:")
    print(f"Cell: {slab.cell}")
    print(f"Surface area: {surface_area_nm2:.4f} nm²")
    
    return slab, surface_area_nm2


def gen_adsorption_sites(slab: Atoms, site_type: str = "ontop") -> List[np.ndarray]:
    """
    Generate adsorption sites on a slab surface.
    
    Parameters:
    - slab: ASE Atoms object of the slab
    - site_type: "ontop", "bridge", "hollow", or "all"
    
    Returns:
    - List of adsorption site coordinates [x, y, z]
    
    Example:
        sites = gen_adsorption_sites(slab, "bridge")
        print(f"Found {len(sites)} bridge sites")
    """
    # Find top layer atoms
    positions = slab.get_positions()
    z_coords = positions[:, 2]
    z_max = np.max(z_coords)
    tolerance = 1e-6
    
    top_layer_mask = np.abs(z_coords - z_max) < tolerance
    top_positions = positions[top_layer_mask]
    
    if len(top_positions) == 0:
        raise ValueError("No atoms found in the top layer")
    
    # Auto-detect characteristic distance (nearest neighbor)
    distance = _get_nn_distance(top_positions, slab.get_cell())
    height = z_max + 2.0  # 2 Å above surface
    
    # Calculate sites based on type
    if site_type == "ontop":
        sites = _calc_ontop_sites(top_positions, height)
    elif site_type == "bridge":
        sites = _calc_bridge_sites(top_positions, distance, height, slab.get_cell())
    elif site_type == "hollow":
        sites = _calc_hollow_sites(top_positions, distance, height, slab.get_cell())
    elif site_type == "all":
        sites = []
        sites.extend(_calc_ontop_sites(top_positions, height))
        sites.extend(_calc_bridge_sites(top_positions, distance, height, slab.get_cell()))
        sites.extend(_calc_hollow_sites(top_positions, distance, height, slab.get_cell()))
    else:
        raise ValueError(f"Unknown site_type: {site_type}")
    
    return sites

def _get_nn_distance(positions: np.ndarray, cell: np.ndarray) -> float:
    """Get nearest neighbor distance automatically."""
    min_dist = float('inf')
    cell_2d = cell[:2, :2] if cell is not None else None
    
    for i in range(len(positions)):
        for j in range(i+1, len(positions)):
            dist = _distance_pbc(positions[i][:2], positions[j][:2], cell_2d)
            if dist > 1e-6:  # Avoid zero distances
                min_dist = min(min_dist, dist)
    
    return min_dist if min_dist != float('inf') else 2.5

def _distance_pbc(pos1: np.ndarray, pos2: np.ndarray, cell: np.ndarray) -> float:
    """Calculate distance with periodic boundary conditions."""
    if cell is None:
        return np.linalg.norm(pos1 - pos2)
    
    diff = pos1 - pos2
    for i in range(2):
        if np.linalg.norm(cell[i]) > 0:
            length = np.linalg.norm(cell[i])
            diff[i] -= length * np.round(diff[i] / length)
    
    return np.linalg.norm(diff)

def _calc_ontop_sites(positions: np.ndarray, height: float) -> List[np.ndarray]:
    """Calculate ontop adsorption sites."""
    sites = []
    for pos in positions:
        site = pos.copy()
        site[2] = height
        sites.append(site)
    return sites

def _calc_bridge_sites(positions: np.ndarray, distance: float, height: float, cell: np.ndarray) -> List[np.ndarray]:
    """Calculate bridge adsorption sites."""
    sites = []
    tolerance = distance * 0.1  # 10% tolerance
    cell_2d = cell[:2, :2] if cell is not None else None
    
    for i in range(len(positions)):
        for j in range(i+1, len(positions)):
            dist = _distance_pbc(positions[i][:2], positions[j][:2], cell_2d)
            
            if abs(dist - distance) < tolerance:
                bridge_site = (positions[i] + positions[j]) / 2
                bridge_site[2] = height
                
                # Check for duplicates
                is_duplicate = any(np.linalg.norm(bridge_site - s) < 0.1 for s in sites)
                if not is_duplicate:
                    sites.append(bridge_site)
    
    return sites

def _calc_hollow_sites(positions: np.ndarray, distance: float, height: float, cell: np.ndarray) -> List[np.ndarray]:
    """Calculate hollow adsorption sites using Delaunay triangulation."""
    sites = []
    tolerance = distance * 0.3  # More lenient for hollow sites
    
    try:
        # Use Delaunay triangulation for robust detection
        tri = Delaunay(positions[:, :2])
        
        for triangle_indices in tri.simplices:
            triangle_pos = positions[triangle_indices]
            
            # Check if triangle has reasonable size
            side_lengths = []
            cell_2d = cell[:2, :2] if cell is not None else None
            for i in range(3):
                j = (i + 1) % 3
                dist = _distance_pbc(triangle_pos[i][:2], triangle_pos[j][:2], cell_2d)
                side_lengths.append(dist)
            
            avg_side = np.mean(side_lengths)
            if abs(avg_side - distance) < tolerance:
                # Calculate centroid
                centroid = np.mean(triangle_pos, axis=0)
                centroid[2] = height
                
                # Check for duplicates
                is_duplicate = any(np.linalg.norm(centroid - s) < 0.1 for s in sites)
                if not is_duplicate:
                    sites.append(centroid)
    
    except Exception:
        # Fallback: simple method
        cell_2d = cell[:2, :2] if cell is not None else None
        for i in range(len(positions)):
            for j in range(i+1, len(positions)):
                for k in range(j+1, len(positions)):
                    # Check if three atoms form reasonable triangle
                    d12 = _distance_pbc(positions[i][:2], positions[j][:2], cell_2d)
                    d23 = _distance_pbc(positions[j][:2], positions[k][:2], cell_2d)
                    d13 = _distance_pbc(positions[i][:2], positions[k][:2], cell_2d)
                    
                    avg_dist = np.mean([d12, d23, d13])
                    if abs(avg_dist - distance) < tolerance:
                        centroid = (positions[i] + positions[j] + positions[k]) / 3
                        centroid[2] = height
                        
                        is_duplicate = any(np.linalg.norm(centroid - s) < 0.1 for s in sites)
                        if not is_duplicate:
                            sites.append(centroid)
    
    return sites

'''
def gen_adsorption_sites(slab: Atoms, nx: int = 3, ny: int = 2, distance: Optional[float] = None, site_type: str = "ontop", atom1_idx: Optional[int] = None, atom2_idx: Optional[int] = None) -> Tuple[Atoms, np.ndarray, List[np.ndarray]]:
    positions = slab.get_positions()
    z_coords = positions[:, 2]
    z_max = np.max(z_coords)
    top_layer_mask = z_coords == z_max
    top_layer_positions = positions[top_layer_mask]
    top_layer_symbols = np.array(slab.get_chemical_symbols())[top_layer_mask]
    
    if len(top_layer_positions) == 0:
        raise ValueError("No atoms found in the top layer of the slab.")
    
    print(f"Number of atoms in top layer at Z={z_max:.4f} Å²: {len(top_layer_positions)}")
    
    nth_x = nx
    nth_y = ny
    sorted_values_x = np.sort(np.unique(np.round(top_layer_positions[:, 0], decimals=10)))
    sorted_values_y = np.sort(np.unique(np.round(top_layer_positions[:, 1], decimals=10)))
    x_target = sorted_values_x[nth_x - 1]
    y_target = sorted_values_y[nth_y - 1]
    print(f"Giá trị X thứ {nth_x} nhỏ nhất: {x_target}")
    print(f"Giá trị Y thứ {nth_y} nhỏ nhất: {y_target}")

    mask = ((top_layer_positions[:, 0] <= x_target) & (top_layer_positions[:, 1] <= y_target))
    selected_positions = top_layer_positions[mask]
    selected_symbols = top_layer_symbols[mask]

    print(f"Các nguyên tử trong top layer với x <= {x_target} và y <= {y_target}:")
    for pos, sym in zip(selected_positions, selected_symbols):
        print(f"{sym}: {pos}")
    
    if distance is not None:
        print(f"Using provided distance: {distance:.8f} Å")
    elif atom1_idx is not None and atom2_idx is not None and 0 <= atom1_idx < len(selected_positions) and 0 <= atom2_idx < len(selected_positions):
        pos1 = selected_positions[atom1_idx][:2]
        pos2 = selected_positions[atom2_idx][:2]
        distance = np.linalg.norm(pos1 - pos2)
        print(f"Khoảng cách XY giữa 2 nguyên tử {pos1} và {pos2}: {distance:.8f} Å (calculated from indices {atom1_idx} and {atom2_idx})")
    else:
        print("Warning: Distance not provided and not enough valid atom indices to calculate. Using default distance of 0.0 Å.")
        distance = 0.0
    
    if len(selected_positions) == 0:
        print("No atoms found with the selected x and y coordinate constraints.")
        return Atoms(), selected_positions, []

    adsorption_sites_list = calculate_adsorption_sites(top_layer_positions, distance, site_type)
    
    print(f"Number of calculated {site_type} adsorption sites: {len(adsorption_sites_list)}")
    print("Adsorption site positions:")
    for i, site in enumerate(adsorption_sites_list):
        print(f"  Site {i}: ({site[0]:.8f}, {site[1]:.8f}, {site[2]:.8f})")
    
    top_layer_atoms = Atoms(
        symbols=top_layer_symbols,
        positions=top_layer_positions,
        cell=slab.get_cell(),
        pbc=slab.get_pbc()
    )
    
    filtered_highlight_atoms = Atoms(
        symbols=['X'] * len(selected_positions),
        positions=selected_positions,
        cell=slab.get_cell(),
        pbc=slab.get_pbc()
    )
    
    combined_atoms = top_layer_atoms + filtered_highlight_atoms
    
    print("Visualizing top layer atoms (original colors) with filtered atoms highlighted using ASE view...")
    
    return combined_atoms, selected_positions, adsorption_sites_list

def calculate_adsorption_sites(top_layer_positions: np.ndarray, distance: float, site_type: str) -> List[np.ndarray]:
    adsorption_sites_list = []
    tolerance = 1e-5

    if site_type == "ontop":
        adsorption_sites_list = [pos for pos in top_layer_positions]
    else:
        for i, pos1 in enumerate(top_layer_positions):
            for j, pos2 in enumerate(top_layer_positions):
                if i < j:
                    dist = np.linalg.norm(pos1[:2] - pos2[:2])
                    if np.isclose(dist, distance, rtol=tolerance, atol=tolerance):
                        if site_type == "bridge":
                            bridge_site = (pos1 + pos2) / 2
                            is_duplicate = False
                            for existing_site in adsorption_sites_list:
                                if np.allclose(bridge_site, existing_site, rtol=tolerance, atol=tolerance):
                                    is_duplicate = True
                                    break
                            if not is_duplicate:
                                adsorption_sites_list.append(bridge_site)
                        
                        elif site_type == "hollow3":
                            for k, pos3 in enumerate(top_layer_positions):
                                if k > j:
                                    dist2 = np.linalg.norm(pos2[:2] - pos3[:2])
                                    dist3 = np.linalg.norm(pos1[:2] - pos3[:2])
                                    if (np.isclose(dist2, distance, rtol=tolerance, atol=tolerance) and
                                        np.isclose(dist3, distance, rtol=tolerance, atol=tolerance)):
                                        hollow3_site = (pos1 + pos2 + pos3) / 3
                                        is_duplicate = False
                                        for existing_site in adsorption_sites_list:
                                            if np.allclose(hollow3_site, existing_site, rtol=tolerance, atol=tolerance):
                                                is_duplicate = True
                                                break
                                        if not is_duplicate:
                                            adsorption_sites_list.append(hollow3_site)

    return adsorption_sites_list
'''

# P, T, Sticking Prob

In [ ]:
def calculate_N_hit(P, A, dt, m, T):
    T = T + 273.15 # Convert Celsius to Kelvin
    m = (m / 1000) / 6.022e23
    k = 1.38e-23
    flux = P / (2 * np.sqrt(2 * np.pi * m * k * T))
    N_hit = int(flux * A * dt)
    print(f"Calculated N_hit: {N_hit} molecules for P={P} Pa, A={A:.4f} m², dt={dt} s, m={m:.2e} kg, T={T} K")
    return N_hit

def generate_and_adsorb_molecules_simplified(slab, adsorption_sites, N_hit, 
                                           existing_molecules, rotated_molecule, 
                                           radius, z_offset=2.0, z_range=(0.5, 5.0), 
                                           MAX_SITES=20, max_no_adsorption=50000, 
                                           molecule_type="precursor",
                                           sticking_probability=0.001):
    
    new_adsorbed_molecules = []
    total_gas_generated = 0
    consecutive_no_adsorption = 0
    total_collision_attempts = 0 
    successful_sticking = 0
    rejected_by_sticking = 0  # New counter
    
    available_sites = adsorption_sites.copy()
    
    if len(available_sites) == 0:
        print(f"   No adsorption sites available for {molecule_type}")
        return [], 0, available_sites
    
    cell = slab.get_cell()
    a1, a2 = cell[0], cell[1]
    slab_z_max = np.max(np.array(available_sites)[:, 2])
    
    pbar = tqdm(total=N_hit, desc=f"Processing {molecule_type}", ncols=80)
    
    while total_gas_generated < N_hit and consecutive_no_adsorption < max_no_adsorption:
        
        if len(available_sites) == 0:
            print(f"   No more available sites for {molecule_type}")
            break
        
        # 1. Generate random position
        r1, r2 = np.random.rand(2)
        xy = r1 * a1[:2] + r2 * a2[:2]
        z = slab_z_max + z_offset + np.random.uniform(z_range[0], z_range[1])
        pos = np.array([xy[0], xy[1], z])
        total_gas_generated += 1
        
        # 2. APPLY STICKING PROBABILITY FIRST (right when molecule hits surface)
        total_collision_attempts += 1
        if np.random.random() >= sticking_probability:
            # Molecule bounces off - doesn't stick
            rejected_by_sticking += 1
            consecutive_no_adsorption += 1
            pbar.update(1)
            continue
        
        # If we reach here, molecule wants to stick - now find a site
        successful_sticking += 1
        
        # 3. Try to find available surface site
        target_positions = np.array(available_sites)
        cu_tree = cKDTree(target_positions[:, :2])
        
        k = min(MAX_SITES, len(available_sites))
        _, indices = cu_tree.query(pos[:2], k=k)
        
        if isinstance(indices, (int, np.integer)):
            indices = [indices]
        
        adsorbed = False
        used_site_idx = None
        
        for site_idx in indices:
            if site_idx >= len(available_sites):
                continue
            
            target_site = available_sites[site_idx]
            
            # Use adsorbed structure for steric check
            test_molecule = rotated_molecule.copy()
            
            target_position = np.array(target_site).copy()
            target_position[2] = slab_z_max + z_offset
            
            offset = target_position - test_molecule.positions[0]
            test_molecule.translate(offset)
            
            # Apply random z-rotation
            test_molecule = rotate_molecule_z_axis(test_molecule)
            
            # Check steric hindrance
            no_overlap = True
            all_existing_molecules = existing_molecules + new_adsorbed_molecules
            
            if all_existing_molecules:
                symbols = []
                positions = []
                for existing_mol in all_existing_molecules:
                    symbols.extend(existing_mol.get_chemical_symbols())
                    positions.extend(existing_mol.get_positions())
                
                if positions:
                    all_existing_atoms = Atoms(symbols=symbols, positions=positions)
                    
                    distances = compute_distances_vectorized(test_molecule.positions, 
                                                           all_existing_atoms.positions)
                    
                    symbols_new = test_molecule.get_chemical_symbols()
                    symbols_existing = all_existing_atoms.get_chemical_symbols()
                    radii_new = np.array([radius[s] for s in symbols_new])
                    radii_existing = np.array([radius[s] for s in symbols_existing])
                    
                    radii_sums = radii_new[:, None] + radii_existing[None, :]
                    if not np.all(distances > radii_sums):
                        no_overlap = False
            
            # If found valid site, adsorb
            if no_overlap:
                new_adsorbed_molecules.append(test_molecule.copy())
                consecutive_no_adsorption = 0
                adsorbed = True
                used_site_idx = site_idx
                break
        
        if adsorbed and used_site_idx is not None:
            available_sites.pop(used_site_idx)
        else:
            # Molecule wanted to stick but no space available
            consecutive_no_adsorption += 1
        
        pbar.update(1)
        if total_gas_generated % 100 == 0:
            actual_efficiency = len(new_adsorbed_molecules) / total_gas_generated
            pbar.set_description(f"{molecule_type}: {len(new_adsorbed_molecules)} ads, "
                               f"{len(available_sites)} sites, eff={actual_efficiency:.4f}")
    
    pbar.close()
    
    if consecutive_no_adsorption >= max_no_adsorption:
        print(f"   Early termination: {consecutive_no_adsorption} consecutive failed attempts")
    
    actual_efficiency = len(new_adsorbed_molecules) / total_gas_generated if total_gas_generated > 0 else 0
    print(f"   {molecule_type.capitalize()}: {total_gas_generated} gas → {len(new_adsorbed_molecules)} adsorbed")
    print(f"   Rejected by sticking probability: {rejected_by_sticking}")
    print(f"   Wanted to stick but no space: {successful_sticking - len(new_adsorbed_molecules)}")
    print(f"   Actual adsorption efficiency: {actual_efficiency:.6f}")
    print(f"   Expected sticking probability: {sticking_probability:.6f}")
    print(f"   Remaining available sites: {len(available_sites)}")
    
    return new_adsorbed_molecules, total_gas_generated, available_sites
'''
def generate_and_adsorb_molecules_simplified(slab, adsorption_sites, N_hit, 
                                           existing_molecules, rotated_molecule, 
                                           radius, z_offset=2.0, z_range=(0.5, 5.0), 
                                           MAX_SITES=20, max_no_adsorption=50000, 
                                           molecule_type="precursor",
                                           sticking_probability=0.001):
    
    new_adsorbed_molecules = []
    total_gas_generated = 0
    consecutive_no_adsorption = 0
    total_collision_attempts = 0 
    successful_sticking = 0
    
    available_sites = adsorption_sites.copy()
    
    if len(available_sites) == 0:
        print(f"   No adsorption sites available for {molecule_type}")
        return [], 0, available_sites
    
    cell = slab.get_cell()
    a1, a2 = cell[0], cell[1]
    slab_z_max = np.max(np.array(available_sites)[:, 2])
    
    pbar = tqdm(total=N_hit, desc=f"Processing {molecule_type}", ncols=80)
    
    while total_gas_generated < N_hit and consecutive_no_adsorption < max_no_adsorption:
        
        if len(available_sites) == 0:
            print(f"   No more available sites for {molecule_type}")
            break
        
        # 1. Generate random position
        r1, r2 = np.random.rand(2)
        xy = r1 * a1[:2] + r2 * a2[:2]
        z = slab_z_max + z_offset + np.random.uniform(z_range[0], z_range[1])
        pos = np.array([xy[0], xy[1], z])
        total_gas_generated += 1
        
        # 2. Try adsorption on available surface sites
        target_positions = np.array(available_sites)
        cu_tree = cKDTree(target_positions[:, :2])
        
        k = min(MAX_SITES, len(available_sites))
        _, indices = cu_tree.query(pos[:2], k=k)
        
        if isinstance(indices, (int, np.integer)):
            indices = [indices]
        
        adsorbed = False
        used_site_idx = None
        
        for site_idx in indices:
            if site_idx >= len(available_sites):
                continue
            
            target_site = available_sites[site_idx]
            
            # Use adsorbed structure for steric check
            test_molecule = rotated_molecule.copy()
            
            target_position = np.array(target_site).copy()
            target_position[2] = slab_z_max + z_offset
            
            offset = target_position - test_molecule.positions[0]
            test_molecule.translate(offset)
            
            # Apply random z-rotation
            test_molecule = rotate_molecule_z_axis(test_molecule)
            
            # 3. Check steric hindrance
            no_overlap = True
            all_existing_molecules = existing_molecules + new_adsorbed_molecules
            
            if all_existing_molecules:
                symbols = []
                positions = []
                for existing_mol in all_existing_molecules:
                    symbols.extend(existing_mol.get_chemical_symbols())
                    positions.extend(existing_mol.get_positions())
                
                if positions:
                    all_existing_atoms = Atoms(symbols=symbols, positions=positions)
                    
                    distances = compute_distances_vectorized(test_molecule.positions, 
                                                           all_existing_atoms.positions)
                    
                    symbols_new = test_molecule.get_chemical_symbols()
                    symbols_existing = all_existing_atoms.get_chemical_symbols()
                    radii_new = np.array([radius[s] for s in symbols_new])
                    radii_existing = np.array([radius[s] for s in symbols_existing])
                    
                    radii_sums = radii_new[:, None] + radii_existing[None, :]
                    if not np.all(distances > radii_sums):
                        no_overlap = False
            
            # 4. Apply sticking probability
            if no_overlap:
                total_collision_attempts += 1
                
                if np.random.random() < sticking_probability:
                    successful_sticking += 1
                    new_adsorbed_molecules.append(test_molecule.copy())
                    consecutive_no_adsorption = 0
                    adsorbed = True
                    used_site_idx = site_idx
                    break
        
        if adsorbed and used_site_idx is not None:
            available_sites.pop(used_site_idx)
        else:
            consecutive_no_adsorption += 1
        
        pbar.update(1)
        if total_gas_generated % 100 == 0:
            collision_efficiency = successful_sticking / total_collision_attempts if total_collision_attempts > 0 else 0
            pbar.set_description(f"{molecule_type}: {len(new_adsorbed_molecules)} ads, "
                               f"{len(available_sites)} sites, eff={collision_efficiency:.4f}")
    
    pbar.close()
    
    if consecutive_no_adsorption >= max_no_adsorption:
        print(f"   Early termination: {consecutive_no_adsorption} consecutive failed attempts")
    
    collision_efficiency = successful_sticking / total_collision_attempts if total_collision_attempts > 0 else 0
    print(f"   {molecule_type.capitalize()}: {total_gas_generated} gas → {len(new_adsorbed_molecules)} adsorbed")
    print(f"   Collision attempts: {total_collision_attempts}, Sticking efficiency: {collision_efficiency:.4f}")
    print(f"   Remaining available sites: {len(available_sites)}")
    
    return new_adsorbed_molecules, total_gas_generated, available_sites

def generate_and_adsorb_molecules_simplified(slab, adsorption_sites, N_hit, 
                                           existing_molecules, rotated_molecule, 
                                           radius, z_offset=2.0, z_range=(0.5, 5.0), 
                                           MAX_SITES=20, max_no_adsorption=50000, 
                                           molecule_type="precursor",
                                           sticking_probability=0.001):
    
    new_adsorbed_molecules = []
    total_gas_generated = 0
    consecutive_no_adsorption = 0
    total_collision_attempts = 0 
    successful_sticking = 0
    
    available_sites = adsorption_sites.copy()
    
    if len(available_sites) == 0:
        print(f"   No adsorption sites available for {molecule_type}")
        return [], 0, available_sites
    
    cell = slab.get_cell()
    a1, a2 = cell[0], cell[1]
    slab_z_max = np.max(np.array(available_sites)[:, 2])
    
    pbar = tqdm(total=N_hit, desc=f"Processing {molecule_type}", ncols=80)
    
    while total_gas_generated < N_hit and consecutive_no_adsorption < max_no_adsorption:
        
        if len(available_sites) == 0:
            print(f"   No more available sites for {molecule_type}")
            break
        
        # 1. Generate random position
        r1, r2 = np.random.rand(2)
        xy = r1 * a1[:2] + r2 * a2[:2]
        z = slab_z_max + z_offset + np.random.uniform(z_range[0], z_range[1])
        pos = np.array([xy[0], xy[1], z])
        total_gas_generated += 1
        
        # 2. Find nearest sites (2D distance)
        target_positions = np.array(available_sites)
        cu_tree = cKDTree(target_positions[:, :2])  # FIX: Only x,y coordinates
        
        k = min(MAX_SITES, len(available_sites))
        _, indices = cu_tree.query(pos[:2], k=k)  # Query with 2D position
        
        if isinstance(indices, (int, np.integer)):
            indices = [indices]
        
        adsorbed = False
        used_site_idx = None
        
        # 3. Try adsorption on nearest sites
        for site_idx in indices:
            if site_idx >= len(available_sites):
                continue
            
            target_site = available_sites[site_idx]
            
            # Use adsorbed structure for steric check
            test_molecule = rotated_molecule.copy()
            
            target_position = np.array(target_site).copy()
            target_position[2] = slab_z_max + z_offset
            offset = target_position - test_molecule.positions[0]
            test_molecule.translate(offset)
            
            # Try multiple z-axis rotations
            n_rotations = 12
            angles = np.linspace(0, 360, n_rotations, endpoint=False)
            
            found_valid_orientation = False
            best_molecule = None
            
            for angle in angles:
                rotated_test = test_molecule.copy()
                pivot = rotated_test.positions[0].copy()
                theta_rad = np.radians(angle)
                
                rotation_matrix = np.array([
                    [np.cos(theta_rad), -np.sin(theta_rad), 0],
                    [np.sin(theta_rad), np.cos(theta_rad), 0],
                    [0, 0, 1]
                ])
                
                rotated_test.positions -= pivot
                rotated_test.positions = np.dot(rotated_test.positions, rotation_matrix.T)
                rotated_test.positions += pivot
                
                # Check steric hindrance
                no_overlap = True
                all_existing_molecules = existing_molecules + new_adsorbed_molecules
                
                if all_existing_molecules:
                    symbols = []
                    positions = []
                    for existing_mol in all_existing_molecules:
                        symbols.extend(existing_mol.get_chemical_symbols())
                        positions.extend(existing_mol.get_positions())
                    
                    if positions:
                        all_existing_atoms = Atoms(symbols=symbols, positions=positions)
                        
                        distances = compute_distances_vectorized(rotated_test.positions, 
                                                               all_existing_atoms.positions)
                        
                        symbols_new = rotated_test.get_chemical_symbols()
                        symbols_existing = all_existing_atoms.get_chemical_symbols()
                        radii_new = np.array([radius[s] for s in symbols_new])
                        radii_existing = np.array([radius[s] for s in symbols_existing])
                        
                        radii_sums = radii_new[:, None] + radii_existing[None, :]
                        if not np.all(distances > radii_sums):
                            no_overlap = False
                
                if no_overlap:
                    found_valid_orientation = True
                    best_molecule = rotated_test.copy()
                    break
            
            # Apply sticking probability
            if found_valid_orientation:
                total_collision_attempts += 1
                
                if np.random.random() < sticking_probability:
                    successful_sticking += 1
                    new_adsorbed_molecules.append(best_molecule)
                    consecutive_no_adsorption = 0
                    adsorbed = True
                    used_site_idx = site_idx
                    break
        
        if adsorbed and used_site_idx is not None:
            available_sites.pop(used_site_idx)
        else:
            consecutive_no_adsorption += 1
        
        pbar.update(1)
        if total_gas_generated % 100 == 0:
            collision_efficiency = successful_sticking / total_collision_attempts if total_collision_attempts > 0 else 0
            pbar.set_description(f"{molecule_type}: {len(new_adsorbed_molecules)} ads, "
                               f"{len(available_sites)} sites, eff={collision_efficiency:.4f}")
    
    pbar.close()
    
    if consecutive_no_adsorption >= max_no_adsorption:
        print(f"   Early termination: {consecutive_no_adsorption} consecutive failed attempts")
    
    collision_efficiency = successful_sticking / total_collision_attempts if total_collision_attempts > 0 else 0
    print(f"   {molecule_type.capitalize()}: {total_gas_generated} gas → {len(new_adsorbed_molecules)} adsorbed")
    print(f"   Collision attempts: {total_collision_attempts}, Sticking efficiency: {collision_efficiency:.4f}")
    print(f"   Remaining available sites: {len(available_sites)}")
    
    return new_adsorbed_molecules, total_gas_generated, available_sites
'''

# Run

In [ ]:
# Simplified ALD Simulation - Direct Surface Adsorption Only
N = 8
P = 1.6
precursor_pulse_time = 7
inhibitor_pulse_time = 120
inert_gas_purge_time = 20
MAX_SITES = 20
z_offset = 2.0
z_range = (2000, 5000.0)
max_no_adsorption = 5000

# Sticking probabilities (define these variables)
STICKING_PROB_PRECURSOR = 1
STICKING_PROB_INHIBITOR = 1

cycle_time = (inhibitor_pulse_time + precursor_pulse_time + 2 * inert_gas_purge_time)  # Removed co_reactant

number_gas_inhibitors = np.array([])
number_ads_inhibitors = np.array([])
number_gas_precursor = np.array([])
number_ads_precursor = np.array([])

# Generate slab and surface area
slab, surface_area_nm2 = generate_slab_from_cif(
    r"C:\Users\dangh\Documents\TaiLieu\ThS Phenikaa\AS-ALD Simulation\CIF Files\SiO2_mp-8352_computed.cif",
    miller_indices=(1, 1, 1), layers=1, vacuum=10, size=(1, 1, 1), repeat=(27, 27, 1)
)
surface_area_m2 = surface_area_nm2 * 1e-18

# Load molecule structures
atoms, radius = atomic_positions(r"C:\Users\dangh\Documents\TaiLieu\ThS Phenikaa\AS-ALD Simulation\Structure File\qe cif file\TiCl4.sdf")
atoms_filtered = atomic_remove_indices(atoms, remove_indices=[1])

inhibitor_atoms, inhibitor_radius = atomic_positions(
    r"C:\Users\dangh\Documents\TaiLieu\ThS Phenikaa\AS-ALD Simulation\Structure File\DMADMS.sdf"
)
inhibitor_atoms_filtered = atomic_remove_indices(inhibitor_atoms, remove_indices=[1,2,5,6,7,8,15,16,17,18,19,20,21,22,23,24,25,26]) 

# Update radius dictionary
radius.update(inhibitor_radius)
radius['O'] = 1.52
radius['Ru'] = 2.3
radius['Ti'] = 2.46  # Å
# Calculate molecular masses and N_hit values
m_precursor = 72.09
m_inhibitor = 62.14
T = 150
N_hit_precursor = calculate_N_hit(P, surface_area_m2, precursor_pulse_time, m_precursor, T)
#N_hit_inhibitor = calculate_N_hit(P, surface_area_m2, inhibitor_pulse_time, m_inhibitor, T)
N_hit_inhibitor = 0
# Generate adsorption sites (only surface sites)
#_, _, adsorption_sites = gen_adsorption_sites(slab, site_type="ontop")
adsorption_sites = gen_adsorption_sites(slab, "hollow")
# Create working copy for site management across cycles
available_sites = adsorption_sites.copy()

# Create rotated molecule templates
selected_site = adsorption_sites[0]
rotated_adsorbate, _, _ = rotate_adsorbate_flexible(atoms_filtered, selected_site, v1_type='centroid', indices=[1, 2, 3])
rotated_inhibitor, _, _ = rotate_adsorbate_flexible(inhibitor_atoms_filtered, selected_site, v1_type='centroid', indices=[1,2])

# Calculate molecular surface areas
_, precursor_area_nm2 = molecule_surface_area(rotated_adsorbate, atom_radii=radius)
_, inhibitor_area_nm2 = molecule_surface_area(rotated_inhibitor, atom_radii=radius)

# Initialize storage for adsorbed molecules
ads_inhibitor_molecules = []
ads_precursor_molecules = []

print(f'Starting ALD simulation with {N} cycles')
print(f'Surface area: {surface_area_nm2:.2f} nm²')
print(f'Initial adsorption sites: {len(available_sites)}')
print(f'N_hit precursor: {N_hit_precursor:.0f}')
print(f'N_hit inhibitor: {N_hit_inhibitor:.0f}')

# Main simulation loop
for cycle in range(1, N+1):
    print(f'\n{"="*50}')
    print(f'CYCLE {cycle}')
    print(f'{"="*50}')
    
    # Phase 1: Inhibitors (only in first cycle)
    if cycle == 1:
        print(f'\n1. INHIBITORS PHASE')
        new_inhibitors, gas_inhibitors_generated, available_sites = generate_and_adsorb_molecules_simplified(
            #atoms=inhibitor_atoms,
            slab=slab,
            adsorption_sites=available_sites,  # Use available sites, not original
            N_hit=N_hit_inhibitor,
            existing_molecules=[],  # No existing molecules in first cycle
            rotated_molecule=rotated_inhibitor,
            radius=radius,
            z_offset=z_offset,
            z_range=z_range,
            MAX_SITES=MAX_SITES,
            max_no_adsorption=max_no_adsorption,
            molecule_type="inhibitor",
            sticking_probability=STICKING_PROB_INHIBITOR
        )
        
        ads_inhibitor_molecules.extend(new_inhibitors)
        number_gas_inhibitors = np.append(number_gas_inhibitors, gas_inhibitors_generated)
        number_ads_inhibitors = np.append(number_ads_inhibitors, len(new_inhibitors))
    else:
        # No inhibitors in subsequent cycles
        number_gas_inhibitors = np.append(number_gas_inhibitors, 0)
        number_ads_inhibitors = np.append(number_ads_inhibitors, 0)
    
    # Phase 2: Precursors (all cycles)
    print(f'\n2. PRECURSORS PHASE')
    
    # All existing molecules (inhibitors + precursors from previous cycles)
    all_existing_molecules = ads_inhibitor_molecules + ads_precursor_molecules
    
    new_precursors, gas_precursors_generated, available_sites = generate_and_adsorb_molecules_simplified(
        #atoms=atoms,
        slab=slab,
        adsorption_sites=available_sites,  # Use updated available sites
        N_hit=N_hit_precursor,
        existing_molecules=all_existing_molecules,
        rotated_molecule=rotated_adsorbate,
        radius=radius,
        z_offset=z_offset,
        z_range=z_range,
        MAX_SITES=MAX_SITES,
        max_no_adsorption=max_no_adsorption,
        molecule_type="precursor",
        sticking_probability=STICKING_PROB_PRECURSOR
    )
    
    # Add newly adsorbed precursors to the collection
    ads_precursor_molecules.extend(new_precursors)
    number_gas_precursor = np.append(number_gas_precursor, gas_precursors_generated)
    number_ads_precursor = np.append(number_ads_precursor, len(new_precursors))
    
    # Cycle summary
    print(f'\n--- CYCLE {cycle} SUMMARY ---')
    if cycle == 1:
        print(f'Inhibitors adsorbed: {len(ads_inhibitor_molecules)}')
    print(f'Precursors adsorbed this cycle: {len(new_precursors)}')
    print(f'Total precursors: {len(ads_precursor_molecules)}')
    print(f'Total molecules in system: {len(ads_inhibitor_molecules) + len(ads_precursor_molecules)}')

# Calculate coverage statistics
total_inhibitors = len(ads_inhibitor_molecules)
total_precursors = len(ads_precursor_molecules)
total_molecules = total_inhibitors + total_precursors

area_coverage_inhibitor = (total_inhibitors * inhibitor_area_nm2) / surface_area_nm2 if surface_area_nm2 > 0 else 0
area_coverage_precursor = (total_precursors * precursor_area_nm2) / surface_area_nm2 if surface_area_nm2 > 0 else 0
total_coverage = area_coverage_inhibitor + area_coverage_precursor

# Final Summary
print(f'\n{"="*60}')
print(f'SIMULATION COMPLETED - FINAL SUMMARY')
print(f'{"="*60}')
print(f'Total Cycles: {N}')
print(f'Total Inhibitors Adsorbed: {total_inhibitors}')
print(f'Total Precursors Adsorbed: {total_precursors}')
print(f'Total Molecules in System: {total_molecules}')
print(f'\nArea Coverage:')
print(f'  Inhibitor: {area_coverage_inhibitor:.4f} ({total_inhibitors} molecules)')
print(f'  Precursor: {area_coverage_precursor:.4f} ({total_precursors} molecules)')
print(f'  Total: {total_coverage:.4f}')

# Collision Efficiency Analysis
print(f'\n{"="*60}')
print(f'COLLISION EFFICIENCY ANALYSIS')
print(f'{"="*60}')
total_gas = np.sum(number_gas_precursor) + np.sum(number_gas_inhibitors)
total_ads = np.sum(number_ads_precursor) + np.sum(number_ads_inhibitors)
overall_efficiency = total_ads / total_gas if total_gas > 0 else 0

print(f'Overall adsorption efficiency: {overall_efficiency:.6f} ({total_ads}/{total_gas})')
print(f'Expected efficiency (precursor): {STICKING_PROB_PRECURSOR:.6f}')
print(f'Expected efficiency (inhibitor): {STICKING_PROB_INHIBITOR:.6f}')

# Individual molecule type efficiencies
precursor_gas = np.sum(number_gas_precursor)
precursor_ads = np.sum(number_ads_precursor)
precursor_efficiency = precursor_ads / precursor_gas if precursor_gas > 0 else 0
print(f'Precursor efficiency: {precursor_efficiency:.6f} ({precursor_ads}/{precursor_gas})')

if np.sum(number_gas_inhibitors) > 0:
    inhibitor_gas = np.sum(number_gas_inhibitors)
    inhibitor_ads = np.sum(number_ads_inhibitors)
    inhibitor_efficiency = inhibitor_ads / inhibitor_gas if inhibitor_gas > 0 else 0
    print(f'Inhibitor efficiency: {inhibitor_efficiency:.6f} ({inhibitor_ads}/{inhibitor_gas})')

# Detailed cycle-by-cycle statistics
print(f'\n📈 DETAILED CYCLE STATISTICS:')
print(f'Sticking Probability Settings:')
print(f'  - Precursor: {STICKING_PROB_PRECURSOR:.6f}')
print(f'  - Inhibitor: {STICKING_PROB_INHIBITOR:.6f}')
print()

for i in range(len(number_gas_inhibitors)):
    cycle_num = i + 1
    print(f'Cycle {cycle_num}:')
    print(f'  - Gas inhibitors: {int(number_gas_inhibitors[i])}')
    print(f'  - Adsorbed inhibitors: {int(number_ads_inhibitors[i])}')
    print(f'  - Gas precursors: {int(number_gas_precursor[i])}')
    print(f'  - Adsorbed precursors: {int(number_ads_precursor[i])}')
    
    if number_gas_inhibitors[i] > 0:
        inh_eff = number_ads_inhibitors[i] / number_gas_inhibitors[i]
        print(f'  - Inhibitor efficiency: {inh_eff:.6f}')
    if number_gas_precursor[i] > 0:
        pre_eff = number_ads_precursor[i] / number_gas_precursor[i]
        print(f'  - Precursor efficiency: {pre_eff:.6f}')
    print()

print(f'{"="*60}')

# Create final structure for visualization
all_adsorbed_molecules = ads_inhibitor_molecules + ads_precursor_molecules
print(f'Total molecules for visualization: {len(all_adsorbed_molecules)}')

if len(all_adsorbed_molecules) > 0:
    all_symbols = []
    all_positions = []
    
    # Add slab atoms
    all_symbols.extend(slab.get_chemical_symbols())
    all_positions.extend(slab.get_positions())
    
    # Add all adsorbed molecules
    for molecule in all_adsorbed_molecules:
        all_symbols.extend(molecule.get_chemical_symbols())
        all_positions.extend(molecule.get_positions())
    
    final_structure = Atoms(symbols=all_symbols, positions=all_positions)
    print('✓ Final structure created successfully for visualization')
else:
    print('⚠️ No molecules were adsorbed. Only slab will be available for visualization.')
    final_structure = slab.copy()

# Visualization
print('\n📊 VISUALIZATION')
print('Displaying final structure with nglview...')
view(final_structure, viewer='ngl')

print('\n🎉 SIMULATION COMPLETED SUCCESSFULLY!')
print(f'Final structure variable: final_structure')
print(f'Total atoms in final structure: {len(final_structure)}')
print(f'Available variables:')
print(f'  - final_structure: Complete structure (slab + molecules)')
print(f'  - ads_inhibitor_molecules: List of adsorbed inhibitor molecules')
print(f'  - ads_precursor_molecules: List of adsorbed precursor molecules')
print(f'  - slab: Original surface slab')
print(f'  - adsorption_sites: Original surface adsorption sites')
print(f'  - available_sites: Remaining available sites ({len(available_sites)} left)')

In [ ]:
view(final_structure, viewer='ngl')

In [ ]:
# Coverage Analysis
print(f'{"="*60}')
print(f'COVERAGE ANALYSIS')
print(f'{"="*60}')

# Calculate coverage (#/nm²)
inhibitor_coverage = total_inhibitors / surface_area_nm2 if surface_area_nm2 > 0 else 0
precursor_coverage = total_precursors / surface_area_nm2 if surface_area_nm2 > 0 else 0
total_coverage_molecules = (total_inhibitors + total_precursors) / surface_area_nm2 if surface_area_nm2 > 0 else 0

print(f'Surface area: {surface_area_nm2:.4f} nm²')
print(f'')
print(f'Coverage of Molecules (#/nm²):')
print(f'  - Inhibitor coverage: {inhibitor_coverage:.4f} molecules/nm²')
print(f'    ({total_inhibitors} molecules / {surface_area_nm2:.4f} nm²)')
print(f'  - Precursor coverage: {precursor_coverage:.4f} molecules/nm²')
print(f'    ({total_precursors} molecules / {surface_area_nm2:.4f} nm²)')
print(f'  - Total coverage: {total_coverage_molecules:.4f} molecules/nm²')
print(f'    ({total_molecules} molecules / {surface_area_nm2:.4f} nm²)')

# Additional analysis: coverage vs theoretical maximum
total_sites = len(adsorption_sites)
sites_per_nm2 = total_sites / surface_area_nm2 if surface_area_nm2 > 0 else 0
site_occupancy = total_molecules / total_sites if total_sites > 0 else 0
coverage_percentage = (total_coverage_molecules / sites_per_nm2) * 100 if sites_per_nm2 > 0 else 0

print(f'')
print(f'Site Analysis:')
print(f'  - Total adsorption sites: {total_sites}')
print(f'  - Maximum theoretical coverage: {sites_per_nm2:.4f} molecules/nm²')
print(f'  - Actual coverage percentage: {coverage_percentage:.2f}%')
print(f'  - Site occupancy: {site_occupancy:.4f} ({site_occupancy*100:.2f}%)')
print(f'  - Available sites remaining: {len(available_sites)} sites')

print(f'{"="*60}')

# Point Coverage Analysis
print(f'\n{"="*60}')
print(f'POINT COVERAGE ANALYSIS')
print(f'{"="*60}')

total_sites = len(adsorption_sites)

# Inhibitor point coverage
inhibitor_point_coverage = (total_inhibitors / total_sites) * 100 if total_sites > 0 else 0
print(f'Inhibitor point coverage: {inhibitor_point_coverage:.2f}%')
print(f'  ({total_inhibitors} molecules / {total_sites} sites)')

# Precursor point coverage
precursor_point_coverage = (total_precursors / total_sites) * 100 if total_sites > 0 else 0
print(f'Precursor point coverage: {precursor_point_coverage:.2f}%')
print(f'  ({total_precursors} molecules / {total_sites} sites)')

# Total point coverage
total_point_coverage = (total_molecules / total_sites) * 100 if total_sites > 0 else 0
print(f'\nTotal point coverage: {total_point_coverage:.2f}%')
print(f'  ({total_molecules} total molecules / {total_sites} sites)')

# Comparison with area coverage
print(f'\nComparison:')
print(f'  - Point coverage: {total_point_coverage:.2f}% (based on site occupancy)')
print(f'  - Area coverage: {total_coverage:.4f} (based on molecular area)')

print(f'{"="*60}')

# Physical Impingement Analysis
print(f'\n{"="*60}')
print(f'PHYSICAL IMPINGEMENT ANALYSIS')
print(f'{"="*60}')

# Calculate physical impingement for each molecule type
precursor_impingement = precursor_gas / surface_area_nm2 if surface_area_nm2 > 0 else 0
print(f'Precursor physical impingement: {precursor_impingement:.2f} #/nm²')
print(f'  ({precursor_gas} molecules / {surface_area_nm2:.4f} nm²)')

if np.sum(number_gas_inhibitors) > 0:
    inhibitor_impingement = inhibitor_gas / surface_area_nm2 if surface_area_nm2 > 0 else 0
    print(f'Inhibitor physical impingement: {inhibitor_impingement:.2f} #/nm²')
    print(f'  ({inhibitor_gas} molecules / {surface_area_nm2:.4f} nm²)')

# Total physical impingement
total_impingement = total_gas / surface_area_nm2 if surface_area_nm2 > 0 else 0
print(f'\nTotal physical impingement: {total_impingement:.2f} #/nm²')
print(f'  ({total_gas} total molecules / {surface_area_nm2:.4f} nm²)')

print(f'{"="*60}')

In [ ]:
# Physical Impingement Analysis
print(f'\n{"="*60}')
print(f'PHYSICAL IMPINGEMENT ANALYSIS')
print(f'{"="*60}')

# Per cycle breakdown
print(f'\nPer-cycle physical impingement:')
for i in range(len(number_gas_precursor)):
    cycle_num = i + 1
    cycle_gas = number_gas_precursor[i] + number_gas_inhibitors[i]
    cycle_impingement = cycle_gas / surface_area_nm2 if surface_area_nm2 > 0 else 0
    
    print(f'Cycle {cycle_num}:')
    if number_gas_inhibitors[i] > 0:
        inh_imp = number_gas_inhibitors[i] / surface_area_nm2
        print(f'  - Inhibitor: {inh_imp:.2f} #/nm² ({int(number_gas_inhibitors[i])} molecules)')
    pre_imp = number_gas_precursor[i] / surface_area_nm2
    print(f'  - Precursor: {pre_imp:.2f} #/nm² ({int(number_gas_precursor[i])} molecules)')
    print(f'  - Cycle total: {cycle_impingement:.2f} #/nm²')

# Cumulative totals
precursor_impingement = np.sum(number_gas_precursor) / surface_area_nm2
inhibitor_impingement = np.sum(number_gas_inhibitors) / surface_area_nm2
total_impingement = (np.sum(number_gas_precursor) + np.sum(number_gas_inhibitors)) / surface_area_nm2

print(f'\nCumulative physical impingement:')
if np.sum(number_gas_inhibitors) > 0:
    print(f'  - Inhibitor: {inhibitor_impingement:.2f} #/nm²')
print(f'  - Precursor: {precursor_impingement:.2f} #/nm²')
print(f'  - Total: {total_impingement:.2f} #/nm²')

print(f'\nSurface area: {surface_area_nm2:.4f} nm²')
print(f'Total gas molecules generated: {int(np.sum(number_gas_precursor) + np.sum(number_gas_inhibitors))}')

print(f'{"="*60}')

In [ ]:
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(15, 15))
# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(label=f'Point Coverage of Inhibitor ({inhibitor_point_coverage:.2f}%)'),
    Patch(label=f'Point Coverage of Precursor ({precursor_point_coverage:.2f}%)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=12)
plot_atoms(final_structure, ax, rotation=('0x,0y,0z'), show_unit_cell=2)
plt.axis('off')
plt.savefig('topview3.png', dpi=300, bbox_inches='tight')
plt.show()